# A/B Testing Analysis: Delivery Speed vs Customer Satisfaction
### Olist E-Commerce Analytics Project

**Objective:** Test whether delivery speed has a statistically significant and practically meaningful effect on customer review scores, using order-level data from the Olist Brazilian e-commerce dataset.

**Hypothesis:** Faster delivery leads to higher customer satisfaction (higher review scores).

## 1. Load Data
Pulling delivery time and review score data from the `vw_delivery_review` PostgreSQL view

In [1]:
import pandas as pd
import psycopg2
from scipy import stats

# Connect to your Postgres database
conn = psycopg2.connect(
    host="localhost",
    database="olist_analytics",
    user="postgres",
    password="54321"  # replace with your actual postgres password
)

# Pull the delivery/review data
query = "SELECT * FROM vw_delivery_review;"
df = pd.read_sql(query, conn)

print(df.shape)
df.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_27820\2867041074.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


(96353, 3)


,order_id,delivery_days,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,4
1,53cdb2fc8bc7dce0b6741e2150273451,13.0,4
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,5
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,5
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0,5


## 2. Bucket Orders by Delivery Speed
Grouping orders into Fast (≤7 days), Medium (8-14 days), and Slow (>14 days) buckets to compare average satisfaction across delivery speed tiers.

In [2]:
# Create delivery speed buckets
def bucket_delivery(days):
    if days <= 7:
        return "Fast (<=7 days)"
    elif days <= 14:
        return "Medium (8-14 days)"
    else:
        return "Slow (>14 days)"

df['delivery_bucket'] = df['delivery_days'].apply(bucket_delivery)

# Quick summary check (should match your SQL Query 6 output)
print(df.groupby('delivery_bucket')['review_score'].agg(['mean', 'count']))

                        mean  count
delivery_bucket                    
Fast (<=7 days)     4.408307  33683
Medium (8-14 days)  4.288446  36395
Slow (>14 days)     3.647954  26275


## 3. Statistical Significance Test (One-Way ANOVA)
Testing whether review scores differ significantly across the three delivery-speed groups.

In [3]:
# Split review scores by group
fast = df[df['delivery_bucket'] == 'Fast (<=7 days)']['review_score']
medium = df[df['delivery_bucket'] == 'Medium (8-14 days)']['review_score']
slow = df[df['delivery_bucket'] == 'Slow (>14 days)']['review_score']

# One-way ANOVA test
f_stat, p_value = stats.f_oneway(fast, medium, slow)

print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value}")

if p_value < 0.05:
    print("Result: Statistically significant difference between groups (p < 0.05)")
else:
    print("Result: No statistically significant difference")

F-statistic: 3081.4740
P-value: 0.0
Result: Statistically significant difference between groups (p < 0.05)


In [4]:
print(f"P-value (scientific notation): {p_value:.4e}")
print(f"P-value (full precision): {p_value:.50f}")

P-value (scientific notation): 0.0000e+00
P-value (full precision): 0.00000000000000000000000000000000000000000000000000


## 4. Pairwise Comparisons
ANOVA confirms *some* group differs — running pairwise t-tests to identify exactly which delivery-speed pairs differ significantly.

In [5]:
# Pairwise t-tests
t1, p1 = stats.ttest_ind(fast, medium)
t2, p2 = stats.ttest_ind(medium, slow)
t3, p3 = stats.ttest_ind(fast, slow)

print(f"Fast vs Medium:  t={t1:.4f}, p={p1}")
print(f"Medium vs Slow:  t={t2:.4f}, p={p2}")
print(f"Fast vs Slow:    t={t3:.4f}, p={p3}")

Fast vs Medium:  t=14.2255, p=7.371109782427355e-46
Medium vs Slow:  t=59.5088, p=0.0
Fast vs Slow:    t=71.0840, p=0.0


## 5. Effect Size (Cohen's d)
With a sample size this large (96K+ orders), even small differences become statistically significant. Effect size tells us whether these differences are *practically* meaningful, not just statistically detectable.

In [6]:
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    pooled_std = ((( n1-1)*var1 + (n2-1)*var2) / (n1+n2-2)) ** 0.5
    return (group1.mean() - group2.mean()) / pooled_std

d_fast_medium = cohens_d(fast, medium)
d_medium_slow = cohens_d(medium, slow)
d_fast_slow = cohens_d(fast, slow)

print(f"Cohen's d — Fast vs Medium: {d_fast_medium:.4f}")
print(f"Cohen's d — Medium vs Slow: {d_medium_slow:.4f}")
print(f"Cohen's d — Fast vs Slow:   {d_fast_slow:.4f}")

Cohen's d — Fast vs Medium: 0.1076
Cohen's d — Medium vs Slow: 0.4817
Cohen's d — Fast vs Slow:   0.5851


**Interpretation:**
- Fast vs Medium: d=0.11 → statistically significant (large sample size), but practically negligible. Going from ≤7 days to 8-14 days barely moves satisfaction.
- Medium vs Slow: d=0.48 → approaching a medium effect — this is where it starts mattering.
- Fast vs Slow: d=0.59 → a medium-to-large effect — the real, actionable gap. Going from fast to slow delivery meaningfully hurts satisfaction.

**Takeaway:** While all pairwise differences are statistically significant due to the large sample size, effect sizes reveal that the real business impact comes from avoiding slow deliveries (15+ days) specifically. The difference between fast and medium delivery is statistically detectable but practically negligible (d=0.11). Operational focus should go toward eliminating the slowest delivery tail, not marginally speeding up already-reasonable delivery times.

Note on experimental design: This is a quasi-experimental (observational) analysis, not a true randomized A/B test. Delivery speed wasn't randomly assigned — it's likely correlated with confounders like customer location (remote areas naturally have both slower delivery AND potentially different baseline satisfaction), order value, or product category. A true experiment would randomly assign delivery speed (e.g., via shipping method) to isolate causation. This analysis establishes a strong association, not proven causation.

## 6. Sample Size / Power Analysis (Bonus)
Simulating: if Olist wanted to run a true randomized A/B test on a delivery-speed intervention, how many customers per group would be needed to reliably detect effects of varying sizes?

In [7]:
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

# Scenario 1: detecting a small effect (d=0.2) - typical for UI/feature experiments
n_small = analysis.solve_power(effect_size=0.2, alpha=0.05, power=0.8, alternative='two-sided')

# Scenario 2: detecting a medium effect (d=0.5) - similar to our Medium vs Slow finding
n_medium = analysis.solve_power(effect_size=0.5, alpha=0.05, power=0.8, alternative='two-sided')

# Scenario 3: detecting our actual observed large effect (d=0.585, Fast vs Slow)
n_large = analysis.solve_power(effect_size=0.585, alpha=0.05, power=0.8, alternative='two-sided')

print(f"Sample size needed per group (small effect, d=0.2):  {n_small:.0f}")
print(f"Sample size needed per group (medium effect, d=0.5): {n_medium:.0f}")
print(f"Sample size needed per group (large effect, d=0.585): {n_large:.0f}")

Sample size needed per group (small effect, d=0.2):  393
Sample size needed per group (medium effect, d=0.5): 64
Sample size needed per group (large effect, d=0.585): 47


## Conclusion

Delivery speed has a statistically significant effect on customer satisfaction (p < 0.001 across all pairwise comparisons), but effect sizes reveal the real business impact is concentrated in avoiding slow deliveries (15+ days) specifically — the Fast vs Slow gap (d=0.585) is medium-to-large, while Fast vs Medium (d=0.108) is negligible in practice.

**Recommendation:** Prioritize eliminating the slowest delivery tail rather than marginally speeding up already-reasonable delivery times.

**Limitation:** This is a quasi-experimental (observational) analysis, not a randomized controlled trial. Delivery speed correlates with confounders like customer location and region, which weren't controlled for. A true experiment would randomly assign shipping method to isolate causation.

**Feasibility for future testing:** With Olist's order volume, detecting even small effect sizes (d=0.2) would require only ~400 customers per group — well within reach of a single week's traffic for most proposed product experiments.